In [ ]:
from tile_server import AggregationStore

In [ ]:
DATABASE_URL = "dbname=testdb user=testuser password=mypassword host=prototyping-pg-1"
TABLE_PREFIX = "v1"
NUM_CLASSES = 5

In [ ]:
import numpy as np

def make_store(level=10):
    return AggregationStore(level=level, database_url=DATABASE_URL, table_prefix=TABLE_PREFIX)

In [ ]:
# Test 1: sum_over_gt=True (default) returns shape (n_pred, n_i, n_j) and class_indices
store = make_store(level=10)
pairs = [(0, 0), (0, 1), (1, 1)]  # gt: {0,1}, pred: {0,1}
result, class_indices = store.read_region(bbox=(0, 0, 2, 3), label_pairs=pairs)
n_pred = len(np.unique([p for _, p in pairs]))
assert result.shape == (n_pred, 3, 4), f"Expected ({n_pred},3,4), got {result.shape}"
# Accept both int32 and int64, since DB fetch may yield int64
assert np.issubdtype(result.dtype, np.integer), f"Expected integer dtype, got {result.dtype}"
print(f"Test 1 passed: sum_over_gt shape={result.shape}, dtype={result.dtype}, class_indices={class_indices}")

In [ ]:
result.dtype

In [ ]:
# Test 2: sum_over_gt=False returns shape (n_gt, n_i, n_j) and class_indices
store = make_store(level=10)
pairs = [(0, 0), (0, 1), (1, 1)]
result, class_indices = store.read_region(bbox=(0, 0, 2, 3), label_pairs=pairs, sum_over_gt=False)
n_gt = len(np.unique([g for g, _ in pairs]))
assert result.shape == (n_gt, 3, 4), f"Expected ({n_gt},3,4), got {result.shape}"
print(f"Test 2 passed: sum_over_gt=False shape={result.shape}, class_indices={class_indices}")

In [ ]:
# Test 3: both sum directions yield the same grand total
store = make_store(level=9)
bbox = (100, 100, 120, 120)
pairs = [(0, 0), (0, 1), (1, 0), (1, 1)]
result_gt, _  = store.read_region(bbox=bbox, label_pairs=pairs, sum_over_gt=True)
result_pred, _ = store.read_region(bbox=bbox, label_pairs=pairs, sum_over_gt=False)
assert result_gt.sum() == result_pred.sum(), \
    f"Grand totals differ: {result_gt.sum()} vs {result_pred.sum()}"
print(f"Test 3 passed: grand total={result_gt.sum()} consistent across both axes")

In [ ]:
# Test 4: all values are non-negative
store = make_store(level=10)
pairs = [(0, 0), (1, 1), (0, 1)]
result, _ = store.read_region(bbox=(0, 0, 9, 9), label_pairs=pairs)
assert (result >= 0).all(), "All patch counts should be non-negative"
print(f"Test 4 passed: shape={result.shape}, per-pred sums={result.sum(axis=(1,2))}")

In [ ]:
# Test 5: grand total matches bbox_search patch_count sum
store = make_store(level=9)
bbox = (100, 100, 150, 150)
all_pairs = [(gt, pred) for gt in range(NUM_CLASSES) for pred in range(NUM_CLASSES)]
result, _ = store.read_region(bbox=bbox, label_pairs=all_pairs)
rows = store.bbox_search(bbox=bbox, label_pairs=all_pairs)
expected = rows[:, 4].sum() if len(rows) > 0 else 0
assert result.sum() == expected, f"Expected {expected}, got {result.sum()}"
print(f"Test 5 passed: matrix total={result.sum()} matches row total={expected}")

In [ ]:
# Test 6: single pred column in sum_over_gt=True matches bbox_search filter
store = make_store(level=9)
bbox = (100, 100, 120, 120)
pairs = [(0, 1), (1, 1)]  # both have pred=1 → only 1 pred column
result, class_indices = store.read_region(bbox=bbox, label_pairs=pairs, sum_over_gt=True)
rows = store.bbox_search(bbox=bbox, label_pairs=pairs)
expected = rows[:, 4].sum() if len(rows) > 0 else 0
assert result.shape[0] == 1  # one unique pred label
assert result.sum() == expected
print(f"Test 6 passed: single pred slice sum={result.sum()}, class_indices={class_indices}")

In [ ]:
# Test 7: close() closes the connection
store = make_store()
store._get_conn()
store.close()
assert store._conn.closed
print("Test 7 passed: close() closes the connection")

In [ ]:
# Test 8: context manager closes connection on exit
store = make_store()
with store as s:
    _ = s.read_region(bbox=(0, 0, 0, 0), label_pairs=[(0, 0)])
assert store._conn.closed
print("Test 8 passed: context manager closes connection on exit")

In [ ]:
# Test 9: table_name is constructed from prefix and level
store = AggregationStore(level=12, database_url=DATABASE_URL, table_prefix="v2")
assert store.table_name == "v2_patch_label_agg_l12"
print(f"Test 9 passed: table_name = '{store.table_name}'")

In [ ]:
print("All tests passed ✓")

In [ ]:
store = make_store(level=10)
bbox = (0, 0, 4, 4)
result_no_mask, class_indices = store.read_region(bbox=bbox, label_pairs=[(0, 0), (1, 1)])

In [ ]:
result_no_mask

In [ ]:
store = make_store(level=9)
bbox = (100, 100, 150, 150)
all_pairs = [(gt, pred) for gt in range(NUM_CLASSES) for pred in range(NUM_CLASSES)]
result_full, class_indices = store.read_region(bbox=bbox, label_pairs=all_pairs)

In [ ]:
result_full.max()

In [ ]:
import time

# Define bounding boxes and aggregation levels to test
bounding_boxes = [
    (0, 0, 10, 10),
    (0, 0, 50, 50),
    (0, 0, 100, 100),
    (0, 0, 500, 500),
    (0, 0, 2000, 2000),
]
levels = [8, 9, 10, 11, 12]

all_pairs = [(gt, pred) for gt in range(NUM_CLASSES) for pred in range(NUM_CLASSES)]

# Measure execution time for each (level, bbox) combination
timings = {}
totals = {}

for level in levels:
    store = make_store(level=level)
    for bbox in bounding_boxes:
        start_time = time.time()
        result, _ = store.read_region(bbox=bbox, label_pairs=all_pairs)
        elapsed = time.time() - start_time
        key = (level, bbox)
        timings[key] = elapsed
        totals[key] = int(result.sum())
        print(f"level={level}  bbox={bbox}  sum={totals[key]}  time={elapsed:.4f}s")
    store.close()

# Print summary table
col_w = 28
header = f"{'(level, bbox)':<{col_w}} {'sum':>12} {'time (s)':>10}"
print("\n" + header)
print("-" * len(header))
for key in timings:
    print(f"{str(key):<{col_w}} {totals[key]:>12} {timings[key]:>10.4f}")

In [ ]:
from tile_server import make_dist_image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

result, class_indices = store.read_region(bbox=(0, 0, 200, 200), label_pairs=all_pairs)

# One color per class (NUM_CLASSES=5)
color_names = ['blue', 'green', 'yellow', 'red', 'purple']
cols = np.array([mcolors.to_rgb(c) for c in color_names], dtype=np.float32)

dist_image = make_dist_image(result, colors=cols, class_indices=class_indices, global_max=np.ones(len(class_indices)))

plt.imshow(dist_image)
plt.title("Distance Image")
plt.savefig("distance_image.png")
print("Test 10 passed: Distance image saved as 'distance_image.png'")

In [ ]:
# Query a different level of the grid
store = make_store(level=8)  # Change the level to 8
result_different_level, class_indices = store.read_region(bbox=(0,0,200,200), label_pairs=all_pairs)

# Print the sum of the result to verify
print(f"Sum of result at level 9: {result_different_level.sum()}")

# One color per class (NUM_CLASSES=5)
color_names = ['blue', 'green', 'yellow', 'red', 'purple']
cols = np.array([mcolors.to_rgb(c) for c in color_names], dtype=np.float32)

dist_image = make_dist_image(result_different_level, colors=cols, class_indices=class_indices, global_max=np.ones(len(class_indices)))

plt.imsave("distance_image.png", dist_image)
print("Test 10 passed: Distance image saved as 'distance_image.png'")


In [ ]:
import holoviews as hv
import matplotlib.colors as mcolors
import numpy as np
from holoviews import streams
from tile_server import make_dist_image

hv.extension('bokeh')

# Color map: one RGB color per predicted class
color_names = ['blue', 'green', 'yellow', 'red', 'purple']
cols = np.array([mcolors.to_rgb(c) for c in color_names], dtype=np.float32)

all_pairs = [(gt, pred) for gt in range(NUM_CLASSES) for pred in range(NUM_CLASSES)]

# World bounds (level-0 pixel coordinates)
WORLD_X_MIN = 0
WORLD_Y_MIN = 0
WORLD_X_MAX = 4096
WORLD_Y_MAX = 4096
WORLD_SIZE = WORLD_X_MAX - WORLD_X_MIN

# Map x/y span → aggregation level.
# Larger spans → coarser level (lower number); smaller spans → finer level.
LEVEL_THRESHOLDS = [
    (2000, 8),
    (500,  9),
    (100, 10),
    (50,  11),
    (0,   12),
]

def choose_level(x_range, y_range):
    if x_range is None or y_range is None:
        return 8
    span = max(x_range[1] - x_range[0], y_range[1] - y_range[0])
    for threshold, level in LEVEL_THRESHOLDS:
        if span > threshold:
            return level
    return LEVEL_THRESHOLDS[-1][1]


def get_image(x_range=None, y_range=None):
    # Fall back to full world bounds when no range is set yet
    x0, x1 = (WORLD_X_MIN, WORLD_X_MAX) if x_range is None else x_range
    y0, y1 = (WORLD_Y_MIN, WORLD_Y_MAX) if y_range is None else y_range

    # Clamp to world bounds
    x0, x1 = max(WORLD_X_MIN, int(x0)), min(WORLD_X_MAX, int(x1))
    y0, y1 = max(WORLD_Y_MIN, int(y0)), min(WORLD_Y_MAX, int(y1))

    # Ensure valid bbox
    if x1 <= x0 or y1 <= y0:
        x0, y0, x1, y1 = WORLD_X_MIN, WORLD_Y_MIN, WORLD_X_MAX, WORLD_Y_MAX

    level = choose_level(x_range, y_range)
    # Convert world coordinates to grid coordinates for the selected level
    grid_scale = 2 ** (12 - level)
    i_min = int(x0 / grid_scale)
    i_max = int(x1 / grid_scale)
    j_min = int(y0 / grid_scale)
    j_max = int(y1 / grid_scale)
    bbox = (i_min, j_min, i_max, j_max)

    store  = make_store(level=level)
    result, class_indices = store.read_region(bbox=bbox, label_pairs=all_pairs)
    store.close()

    rgb = make_dist_image(result, colors=cols, class_indices=class_indices, global_max=np.ones(len(class_indices)))          # (H, W, 3) float32 [0,1]

    print(f"level={level}  bbox={bbox}  shape={rgb.shape}  sum={result.sum()}")

    # hv.RGB expects bounds as (left, bottom, right, top) in world coordinates
    return hv.RGB(rgb, bounds=(x0, y0, x1, y1)).opts(
        width=600, height=600,
        xaxis='bottom', yaxis='left',
        title=f"Level {level} | bbox {bbox}",
        tools=['box_zoom', 'reset', 'pan'],
    )


range_stream = streams.RangeXY()

dmap = hv.DynamicMap(get_image, streams=[range_stream])
dmap.opts(width=600, height=600)
